# Homework 3 — Training pix2pixHD on BBBC010

We use NVIDIA's [pix2pixHD](https://github.com/NVIDIA/pix2pixHD) on the BBBC010 data. The dataset has 80 train pairs (mask → brightfield) at
512×512. We train for 40 epochs and save **milestone checkpoints** at epochs 5, 10, 20, 40
so the evaluation notebook can compare quality over time.

## 1. Clone the original pix2pixHD repo

In [1]:
!git clone https://github.com/NVIDIA/pix2pixHD.git

fatal: destination path 'pix2pixHD' already exists and is not an empty directory.


## 2. Apply the Python-3.11+ patches

The original repo targeted Python 3.5 / PyTorch 0.4. `apply_patches.py` (sitting next to
`pix2pixHD/`) makes three small changes

In [2]:
!python apply_patches.py

  skip  pix2pixHD/train.py (already patched)
  skip  pix2pixHD/models/networks.py (already patched)
  skip  pix2pixHD/models/pix2pixHD_model.py (already patched)
  skip  pix2pixHD/options/base_options.py (already patched)
  skip  pix2pixHD/models/pix2pixHD_model.py (already patched)
All patches applied.


## 3. Inspect data layout

After running `00_PrepData.ipynb` you should have:
```
datasets/bbbc010_pix2pixhd/
├── train_A/   # 80 binary masks (RGB, 512×512)
├── train_B/   # 80 brightfield images
├── test_A/    # 20 masks
└── test_B/    # 20 images
```
Because we use `--label_nc 0`, pix2pixHD treats the input as a raw image, not a semantic map.

## 4. Training command

Same flags as Unit 5. **Milestone checkpoints** (epochs 5/10/20/40) come from
`apply_patches.py` — they let the evaluation notebook show how generation quality evolves.

Expect ~1–2 hours on a single modern GPU. Adjust `--batchSize` to fit your memory.

In [1]:
!cd pix2pixHD && python train.py \
    --name bbbc010_512 \
    --dataroot ../datasets/bbbc010_pix2pixhd \
    --label_nc 0 \
    --no_instance \
    --loadSize 512 \
    --fineSize 512 \
    --batchSize 2 \
    --niter 100 \
    --niter_decay 0 \
    --save_epoch_freq 100 \
    --gpu_ids 0 \
    --checkpoints_dir ./checkpoints

------------ Options -------------
batchSize: 2
beta1: 0.5
checkpoints_dir: ./checkpoints
continue_train: False
data_type: 32
dataroot: ../datasets/bbbc010_pix2pixhd
debug: False
display_freq: 100
display_winsize: 512
feat_num: 3
fineSize: 512
fp16: False
gpu_ids: [0]
input_nc: 3
instance_feat: False
isTrain: True
label_feat: False
label_nc: 0
lambda_feat: 10.0
loadSize: 512
load_features: False
load_pretrain: 
local_rank: 0
lr: 0.0002
max_dataset_size: inf
model: pix2pixHD
nThreads: 2
n_blocks_global: 9
n_blocks_local: 3
n_clusters: 10
n_downsample_E: 4
n_downsample_global: 4
n_layers_D: 3
n_local_enhancers: 1
name: bbbc010_512
ndf: 64
nef: 16
netG: global
ngf: 64
niter: 100
niter_decay: 0
niter_fix_global: 0
no_flip: False
no_ganFeat_loss: False
no_html: False
no_instance: True
no_lsgan: False
no_vgg_loss: False
norm: instance
num_D: 2
output_nc: 3
phase: train
pool_size: 0
print_freq: 100
resize_or_crop: scale_width
save_epoch_freq: 100
save_latest_freq: 1000
serial_batches: False
t

## 5. Verify checkpoints

After training, you should see `5_net_*.pth`, `10_net_*.pth`, `20_net_*.pth`,
`40_net_*.pth` plus `latest_net_*.pth` in `pix2pixHD/checkpoints/bbbc010_512/`.

In [2]:
!ls -lthr pix2pixHD/checkpoints/bbbc010_512/

total 5.7G
drwxr-xr-x 1 vscode vscode  512 Jun  8 12:15 web
-rw-r--r-- 1 vscode vscode 1.1K Jun  8 12:39 opt.txt
-rw-r--r-- 1 vscode vscode 696M Jun  8 12:41 5_net_G.pth
-rw-r--r-- 1 vscode vscode  22M Jun  8 12:41 5_net_D.pth
-rw-r--r-- 1 vscode vscode 696M Jun  8 12:42 10_net_G.pth
-rw-r--r-- 1 vscode vscode  22M Jun  8 12:42 10_net_D.pth
-rw-r--r-- 1 vscode vscode 696M Jun  8 12:44 20_net_G.pth
-rw-r--r-- 1 vscode vscode  22M Jun  8 12:44 20_net_D.pth
-rw-r--r-- 1 vscode vscode 696M Jun  8 12:49 40_net_G.pth
-rw-r--r-- 1 vscode vscode  22M Jun  8 12:49 40_net_D.pth
-rw-r--r-- 1 vscode vscode 696M Jun  8 12:53 60_net_G.pth
-rw-r--r-- 1 vscode vscode  22M Jun  8 12:53 60_net_D.pth
-rw-r--r-- 1 vscode vscode 696M Jun  8 12:57 80_net_G.pth
-rw-r--r-- 1 vscode vscode  22M Jun  8 12:57 80_net_D.pth
-rw-r--r-- 1 vscode vscode  13K Jun  8 13:01 loss_log.txt
-rw-r--r-- 1 vscode vscode 696M Jun  8 13:01 latest_net_G.pth
-rw-r--r-- 1 vscode vscode  22M Jun  8 13:01 latest_net_D.pth
-rw-r--r-- 

## Conclusion

Mask-conditioned brightfield image generation: the model has to fill in
plausible worm textures inside the mask outline while leaving the background empty.
Next: `02_Evaluation.ipynb` quantifies how well it learned this over the 40-epoch run.